# Medical LLM QLoRA cloud runner

This notebook contains no training logic. It clones the repository, optionally persists outputs to Google Drive, reads an optional `HF_TOKEN` from notebook secrets, runs a 10-step smoke test, then runs the measured pipeline. Use a GPU runtime.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/Leng-Bu-Ding/medical-llm-qlora.git'
base = Path('/content') if Path('/content').exists() else Path('/kaggle/working')
repo = base / 'medical-llm-qlora'
if (repo / '.git').exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
os.chdir(repo)
print(repo)

In [ ]:
# Recommended for multi-hour Colab runs. Set True and approve the Drive mount
# before training so checkpoints survive runtime reclamation.
from pathlib import Path
repo = Path.cwd()

PERSIST_OUTPUTS_TO_DRIVE = True
if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    persistent_root = Path('/content/drive/MyDrive/medical-llm-qlora')
    persistent_outputs = persistent_root / 'outputs'
    persistent_outputs.mkdir(parents=True, exist_ok=True)
    local_outputs = repo / 'outputs'
    if local_outputs.exists() and not local_outputs.is_symlink():
        raise RuntimeError('Enable Drive persistence before creating local outputs.')
    if not local_outputs.exists():
        local_outputs.symlink_to(persistent_outputs, target_is_directory=True)
    print('Persistent outputs:', persistent_outputs)
else:
    print('Outputs are ephemeral and will be lost when the runtime is reclaimed.')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-train.txt'], check=True)

In [ ]:
# Optional read-only Hugging Face token. Never print or commit it.
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if token:
    os.environ['HF_TOKEN'] = token
print('HF_TOKEN configured:', bool(token))

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GiB')

In [ ]:
# Mandatory 10-step validation before spending hours on the full run.
# subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--run-id', 'smoke_clean', '--protocol', 'clean', '--smoke'], check=True)

In [ ]:
# Change to True only after the smoke test succeeds.
RUN_FULL = True
RUN_ID = 'clean_main_v1'
if RUN_FULL:
    subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--run-id', RUN_ID, '--protocol', 'clean'], check=True)

In [ ]:
# Optional post-core external transfer check and rank ablation.
RUN_ENHANCEMENTS = False
if RUN_ENHANCEMENTS:
    subprocess.run([sys.executable, 'scripts/run_external_eval.py', '--adapter', f'outputs/{RUN_ID}/training/adapter', '--output-dir', f'outputs/{RUN_ID}'], check=True)
    subprocess.run([sys.executable, 'scripts/run_ablation.py', '--data-dir', f'outputs/{RUN_ID}/data', '--output-dir', 'outputs/ablation_rank'], check=True)

In [ ]:
# Archive outputs before the hosted runtime is reclaimed.
archive = subprocess.check_output([sys.executable, '-c', "import shutil; print(shutil.make_archive('medical_qlora_outputs', 'zip', 'outputs'))"], text=True).strip()
print('Download this file:', archive)